# Flickr30k Data Contract

Notebook CPU khóa dữ liệu Flickr30k trước khi chạy baseline/KD.

## Input bắt buộc

1. Dataset ảnh Flickr30k, chứa thư mục ảnh `.jpg`.
2. `dataset_flickr30k.json` theo Karpathy split. `captions.txt` đơn lẻ
   **không đủ**, vì không chứa nhãn Train/Validation/Test chuẩn.

## Output

- `f30k_train.parquet`: caption Train/Restval đã token hóa.
- `f30k_validation.parquet`: caption Validation.
- `f30k_image_manifest.parquet`: đường dẫn ảnh Train/Validation.
- `f30k_test_sealed.parquet`: chỉ image ID/path, không chứa caption.
- `f30k_vocabulary_minfreq5.json`.
- `F30K_DATA_CONTRACT.json`.
- Manifest và SHA-256 toàn bộ output.

Runtime: CPU, Accelerator None, Internet Off; dự kiến 20–60 phút.


## Cell 1 — Khám phá input và khóa môi trường


In [1]:
import csv
import hashlib
import json
import os
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError as exc:
    raise RuntimeError("Kaggle image phải có pyarrow") from exc

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUTPUT_ROOT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT", "/kaggle/working/F30K_Data"
))
PARQUET_DIR = OUTPUT_ROOT / "parquet"
CONFIG_DIR = OUTPUT_ROOT / "configs"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in (OUTPUT_ROOT, PARQUET_DIR, CONFIG_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

annotation_matches = sorted(INPUT_ROOT.rglob("dataset_flickr30k.json"))
assert len(annotation_matches) == 1, (
    "Cần đúng một dataset_flickr30k.json theo Karpathy split. "
    "Input chỉ có Images/ và captions.txt chưa đủ vì captions.txt không "
    "chứa nhãn Train/Validation/Test. Hãy gắn thêm dataset/split JSON; "
    "không tự chia ngẫu nhiên để tránh làm sai thiết kế thực nghiệm.",
    annotation_matches,
)
ANNOTATION_PATH = annotation_matches[0]

image_candidates = [
    path for path in INPUT_ROOT.rglob("*.jpg")
    if "flickr" in path.as_posix().lower()
]
assert len(image_candidates) >= 30_000, (
    "Không tìm đủ ảnh Flickr30k", len(image_candidates)
)

# Kaggle inputs can expose more than one complete Flickr30k image tree.  Hashing
# every duplicate would reread several gigabytes and can make Cell 1 look
# stalled.  Select one complete image directory deterministically instead.
IMAGE_PATHS_BY_PARENT = {}
IMAGE_PATHS_BY_NAME = {}
for path in image_candidates:
    IMAGE_PATHS_BY_PARENT.setdefault(path.parent, []).append(path)
    IMAGE_PATHS_BY_NAME.setdefault(path.name, []).append(path)

complete_image_roots = sorted(
    (
        (parent, paths)
        for parent, paths in IMAGE_PATHS_BY_PARENT.items()
        if len({path.name for path in paths}) >= 30_000
    ),
    key=lambda item: (-len({path.name for path in item[1]}), item[0].as_posix()),
)
assert complete_image_roots, (
    "Không tìm thấy một thư mục ảnh Flickr30k hoàn chỉnh",
    {
        parent.as_posix(): len({path.name for path in paths})
        for parent, paths in IMAGE_PATHS_BY_PARENT.items()
    },
)
SELECTED_IMAGE_ROOT, selected_image_candidates = complete_image_roots[0]
IMAGE_BY_NAME = {}
for path in sorted(selected_image_candidates, key=lambda value: value.as_posix()):
    assert path.name not in IMAGE_BY_NAME, (
        "Trùng tên ảnh bên trong thư mục nguồn đã chọn",
        path.name,
        SELECTED_IMAGE_ROOT,
    )
    IMAGE_BY_NAME[path.name] = path

DUPLICATE_IMAGE_NAME_COUNT = sum(
    len(paths) > 1 for paths in IMAGE_PATHS_BY_NAME.values()
)

print("Annotation :", ANNOTATION_PATH)
print("JPG paths  :", len(image_candidates))
print("JPG unique :", len(IMAGE_BY_NAME))
print("Image root :", SELECTED_IMAGE_ROOT)
print("Other roots:", max(0, len(complete_image_roots) - 1), "ignored")
print("Duplicates :", DUPLICATE_IMAGE_NAME_COUNT, "names across all inputs")
print("Output     :", OUTPUT_ROOT)


Annotation : /kaggle/input/datasets/shtvkumar/karpathy-splits/dataset_flickr30k.json
JPG paths  : 63566
JPG unique : 31783
Image root : /kaggle/input/datasets/hsankesara/flickr-image-dataset/flickr30k_images/flickr30k_images
Other roots: 1 ignored
Duplicates : 31783 names across all inputs
Output     : /kaggle/working/F30K_Data


## Cell 2 — Khóa split, tokenizer và vocabulary


In [2]:
raw = json.loads(ANNOTATION_PATH.read_text(encoding="utf-8"))
images = raw["images"] if isinstance(raw, dict) else raw
assert isinstance(images, list) and images

TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?|[^\w\s]", re.UNICODE)
MAX_LEXICAL_TOKENS = 29  # + BOS/EOS = 31 decoder positions.
MIN_FREQUENCY = 5

def lexical_tokens(caption):
    return [token.lower() for token in TOKEN_RE.findall(caption.strip())]

split_counts = Counter()
caption_counts = Counter()
missing_images = []
for item in images:
    split = str(item["split"]).lower()
    split_counts[split] += 1
    filename = item["filename"]
    if filename not in IMAGE_BY_NAME:
        missing_images.append(filename)
    if split in {"train", "restval"}:
        for sentence in item.get("sentences", []):
            raw_caption = sentence.get("raw", "")
            caption_counts.update(lexical_tokens(raw_caption))

assert not missing_images, (
    "Dataset thiếu ảnh được khai báo trong JSON",
    missing_images[:20],
    len(missing_images),
)
assert sum(split_counts.values()) == len(images)
assert split_counts["test"] > 0 and split_counts["val"] > 0
assert split_counts["train"] + split_counts["restval"] > 0

special = ["<pad>", "<bos>", "<eos>", "<unk>"]
vocabulary_tokens = special + sorted(
    token for token, count in caption_counts.items()
    if count >= MIN_FREQUENCY and token not in special
)
token_to_id = {token: index for index, token in enumerate(vocabulary_tokens)}
assert [token_to_id[token] for token in special] == [0, 1, 2, 3]

VOCAB_PATH = CONFIG_DIR / "f30k_vocabulary_minfreq5.json"
write_json(VOCAB_PATH, {
    "schema_version": "f30k_vocab_v1",
    "created_at_utc": now_utc(),
    "minimum_frequency": MIN_FREQUENCY,
    "max_lexical_tokens": MAX_LEXICAL_TOKENS,
    "special_token_ids": {"pad": 0, "bos": 1, "eos": 2, "unk": 3},
    "tokens": vocabulary_tokens,
    "token_to_id": token_to_id,
})

def encode(caption):
    tokens = lexical_tokens(caption)
    retained = tokens[:MAX_LEXICAL_TOKENS]
    ids = [1] + [token_to_id.get(token, 3) for token in retained] + [2]
    return tokens, retained, ids

print("Split counts :", dict(split_counts))
print("Vocabulary   :", len(vocabulary_tokens))
print("Vocab SHA256 :", sha256(VOCAB_PATH))


Split counts : {'train': 29000, 'test': 1000, 'val': 1014}
Vocabulary   : 7456
Vocab SHA256 : c94727d24d24130126ae7c88d9537714b88ff3dde12ac741f9cd4a0757287a83


## Cell 3 — Xuất parquet và niêm phong Test


In [3]:
caption_schema = pa.schema([
    pa.field("annotation_id", pa.int64(), nullable=False),
    pa.field("image_id", pa.int64(), nullable=False),
    pa.field("karpathy_split", pa.string(), nullable=False),
    pa.field("source_partition", pa.string(), nullable=False),
    pa.field("caption", pa.string(), nullable=False),
    pa.field("token_ids", pa.list_(pa.int32()), nullable=False),
    pa.field("original_lexical_length", pa.int16(), nullable=False),
    pa.field("retained_lexical_length", pa.int16(), nullable=False),
    pa.field("was_truncated", pa.bool_(), nullable=False),
])

manifest_schema = pa.schema([
    pa.field("image_id", pa.int64(), nullable=False),
    pa.field("karpathy_split", pa.string(), nullable=False),
    pa.field("experiment_split", pa.string(), nullable=False),
    pa.field("source_partition", pa.string(), nullable=False),
    pa.field("image_path", pa.string(), nullable=False),
])

test_schema = pa.schema([
    pa.field("image_id", pa.int64(), nullable=False),
    pa.field("karpathy_split", pa.string(), nullable=False),
    pa.field("experiment_split", pa.string(), nullable=False),
    pa.field("source_partition", pa.string(), nullable=False),
    pa.field("image_path", pa.string(), nullable=False),
    pa.field("caption_included", pa.bool_(), nullable=False),
    pa.field("test_set_status", pa.string(), nullable=False),
])

train_rows, validation_rows, manifest_rows, test_rows = [], [], [], []
annotation_id = 0
for fallback_image_id, item in enumerate(images):
    split = str(item["split"]).lower()
    filename = item["filename"]
    numeric = re.findall(r"\d+", Path(filename).stem)
    image_id = int(numeric[-1]) if numeric else fallback_image_id
    image_path = IMAGE_BY_NAME[filename]
    experiment_split = (
        "train" if split in {"train", "restval"}
        else "validation" if split == "val"
        else "test"
    )
    if experiment_split == "test":
        test_rows.append({
            "image_id": image_id,
            "karpathy_split": split,
            "experiment_split": "test",
            "source_partition": "flickr30k",
            "image_path": str(image_path),
            "caption_included": False,
            "test_set_status": "sealed",
        })
        continue
    manifest_rows.append({
        "image_id": image_id,
        "karpathy_split": split,
        "experiment_split": experiment_split,
        "source_partition": "flickr30k",
        "image_path": str(image_path),
    })
    destination = train_rows if experiment_split == "train" else validation_rows
    for sentence in item.get("sentences", []):
        caption = str(sentence.get("raw", "")).strip()
        original, retained, ids = encode(caption)
        destination.append({
            "annotation_id": annotation_id,
            "image_id": image_id,
            "karpathy_split": split,
            "source_partition": "flickr30k",
            "caption": caption,
            "token_ids": ids,
            "original_lexical_length": len(original),
            "retained_lexical_length": len(retained),
            "was_truncated": len(original) > len(retained),
        })
        annotation_id += 1

paths = {
    "train": PARQUET_DIR / "f30k_train.parquet",
    "validation": PARQUET_DIR / "f30k_validation.parquet",
    "image_manifest": PARQUET_DIR / "f30k_image_manifest.parquet",
    "test_sealed": PARQUET_DIR / "f30k_test_sealed.parquet",
}
pq.write_table(pa.Table.from_pylist(train_rows, schema=caption_schema), paths["train"])
pq.write_table(pa.Table.from_pylist(validation_rows, schema=caption_schema), paths["validation"])
pq.write_table(pa.Table.from_pylist(manifest_rows, schema=manifest_schema), paths["image_manifest"])
pq.write_table(pa.Table.from_pylist(test_rows, schema=test_schema), paths["test_sealed"])

assert pq.ParquetFile(paths["train"]).metadata.num_rows == len(train_rows)
assert pq.ParquetFile(paths["validation"]).metadata.num_rows == len(validation_rows)
assert pq.ParquetFile(paths["test_sealed"]).metadata.num_rows == len(test_rows)
assert all(not row["caption_included"] for row in test_rows)

print("Train captions      :", len(train_rows))
print("Validation captions :", len(validation_rows))
print("Train/val images    :", len(manifest_rows))
print("Test images sealed  :", len(test_rows))


Train captions      : 145000
Validation captions : 5070
Train/val images    : 30014
Test images sealed  : 1000


## Cell 4 — Contract, SHA-256 và manifest


In [4]:
CONTRACT_PATH = OUTPUT_ROOT / "F30K_DATA_CONTRACT.json"
contract = {
    "schema_version": "f30k_data_contract_v1",
    "created_at_utc": now_utc(),
    "status": "complete",
    "dataset": "Flickr30k",
    "annotation_source": str(ANNOTATION_PATH),
    "annotation_sha256": sha256(ANNOTATION_PATH),
    "image_count_json": len(images),
    "image_path_count_discovered": len(image_candidates),
    "image_count_available": len(IMAGE_BY_NAME),
    "selected_image_root": SELECTED_IMAGE_ROOT.as_posix(),
    "duplicate_image_name_count": DUPLICATE_IMAGE_NAME_COUNT,
    "duplicate_policy": "single_complete_image_root_selected_deterministically",
    "split_counts_raw": dict(split_counts),
    "train_image_count": split_counts["train"] + split_counts["restval"],
    "validation_image_count": split_counts["val"],
    "test_image_count": split_counts["test"],
    "train_caption_count": len(train_rows),
    "validation_caption_count": len(validation_rows),
    "vocabulary_size": len(vocabulary_tokens),
    "vocabulary_sha256": sha256(VOCAB_PATH),
    "max_decoder_steps": MAX_LEXICAL_TOKENS + 2,
    "test_set_status": "sealed",
    "test_caption_exported": False,
    "artifacts": {
        name: {
            "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for name, path in {**paths, "vocabulary": VOCAB_PATH}.items()
    },
}
write_json(CONTRACT_PATH, contract)

artifact_paths = sorted(
    path for path in OUTPUT_ROOT.rglob("*")
    if path.is_file() and "manifest" not in path.name.lower()
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
    for path in artifact_paths
]
MANIFEST_CSV = MANIFEST_DIR / "F30K_MANIFEST.csv"
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(MANIFEST_DIR / "F30K_MANIFEST.json", {
    "schema_version": "f30k_manifest_v1",
    "created_at_utc": now_utc(),
    "artifact_count": len(manifest_rows),
    "artifacts": manifest_rows,
})

for row in manifest_rows:
    path = OUTPUT_ROOT / row["relative_path"]
    assert path.stat().st_size == row["size_bytes"]
    assert sha256(path) == row["sha256"]

print("=" * 72)
print("FLICKR30K DATA CONTRACT: COMPLETE")
print("Output        :", OUTPUT_ROOT)
print("Train images  :", contract["train_image_count"])
print("Val images    :", contract["validation_image_count"])
print("Test images   :", contract["test_image_count"], "SEALED")
print("Vocabulary    :", contract["vocabulary_size"])
print("Manifest rows :", len(manifest_rows))


FLICKR30K DATA CONTRACT: COMPLETE
Output        : /kaggle/working/F30K_Data
Train images  : 29000
Val images    : 1014
Test images   : 1000 SEALED
Vocabulary    : 7456
Manifest rows : 5
